# Training Dueling DQN Agents

This notebook trains the `DQNVec11Agent` and `DQNGridAgent` utilizing Dueling DQN architecture on vectorized environments. Dueling DQN separates state values from action advantages, drastically speeding up learning in collision-heavy environments like Snake.


In [3]:
import sys
import os
import torch

sys.path.append(os.path.abspath(".."))  # Ensure imports from core/agents work

from gymnasium.vector import SyncVectorEnv, AsyncVectorEnv
from gymnasium.wrappers import FrameStackObservation

from core.env.core import SnakeEnv
from core.env.types import ObserveType, RewardOptions
from agents.dqn.dqn_vec11 import DQNVec11Agent
from agents.dqn.dqn_grid import DQNGridAgent

## Environment Setup

Reward shaping is key. We apply heavy penalties for collisions, and give small dense rewards for making steps closer to the apple.


In [4]:
def make_env(env_id, obs_type, width=20, height=20, frame_stack=1):
    def _init():
        env = SnakeEnv(
            width=width,
            height=height,
            obs_type=obs_type,
            num_apples=3,
            num_obstacles=15,
            max_steps=7500,
            seed=42 + env_id,
            reward_options=RewardOptions(
                eats_apple=15.0,
                complete=100.0,
                penalty_step=-0.0005,
                penalty_loop=-3.0,
                death_wall=-8.0,
                death_self=-10.0,
                shaping_closer=0.5,
                shaping_further=-0.2,
            ),
        )
        if frame_stack > 1:
            env = FrameStackObservation(env, stack_size=frame_stack)
        return env

    return _init


num_envs = 16

## 1. Train Vec11 Agent

The 11-dimensional observation vector uses linear layers and usually learns extremely quickly.


In [ ]:
envs_vec = SyncVectorEnv(make_env(i, ObserveType.VEC_11) for i in range(8))

agent_vec11 = DQNVec11Agent(
    learning_rate=8e-4, weight_decay=0.001, device="cuda", model_file="../artifacts/models/dqn_vec11_v4.pth"
)

print("Training Vec11 Agent...")
agent_vec11.train(
    env=envs_vec,
    total_timesteps=200_000,  # × 8 = 1.6M effective; more than before
    log_step=5_000,
    buffer_size=50_000,  # larger: needs more diverse long-episode samples
    batch_size=512,
    gamma=0.995,  # was 0.99 — higher gamma critical for long-horizon planning
    tau=0.008,
    eps_init=1.0,
    eps_final=0.005,  # very low: squeeze maximum exploitation
    eps_decay=0.06,  # fast explore then long exploit phase
    learning_starts=1_000,
    train_freq=1,
    gradient_steps=6,
)

Training Vec11 Agent...


Parallel Training: 100%|██████████| 200000/200000 [32:18<00:00, 103.16it/s, Avg Rwd (100)=95.63, Best=99.62, Eps=0.005]


{'episode_rewards': [-20.11,
  -20.11,
  -20.13,
  -20.24,
  -19.95,
  -14.860000000000001,
  -20.41,
  -19.81,
  -20.78,
  -20.11,
  -20.37,
  -20.32,
  -19.82,
  -20.06,
  -21.0,
  -21.75,
  -15.730000000000004,
  -20.18,
  -14.910000000000005,
  -20.1,
  -20.73,
  -21.54,
  -9.919999999999995,
  -21.16,
  -20.13,
  -19.79,
  -19.66,
  -21.6,
  -21.79,
  -19.99,
  -20.06,
  -22.03,
  -22.11,
  -15.310000000000002,
  -20.02,
  -20.13,
  -20.91,
  -22.09,
  -19.66,
  -9.8,
  -21.43,
  -20.11,
  -21.83,
  -20.85,
  -20.02,
  -20.47,
  -22.62,
  -19.04,
  -5.449999999999999,
  -20.45,
  -19.93,
  -21.02,
  -22.25,
  -21.49,
  -26.650000000000002,
  -21.85,
  -21.76,
  -21.3,
  -19.9,
  -20.11,
  -20.0,
  -20.39,
  -20.16,
  -14.660000000000002,
  -11.2,
  -20.02,
  -20.92,
  -19.77,
  -20.02,
  -20.56,
  -22.22,
  -21.23,
  -23.029999999999998,
  -20.37,
  -20.84,
  -20.26,
  -21.2,
  -23.709999999999997,
  -20.93,
  -21.68,
  -20.06,
  -20.24,
  -14.710000000000003,
  -23.54,
  -15.8100

## 2. Train Grid Agent (CNN)

The CNN agent uses a 3D grid observation (channels for snake, apples, obstacles). We use frame stacking (`frame_stack=2`) so the network can infer movement direction from consecutive frames. CNNs generally take longer to learn spatial features, so we increase `total_timesteps`.


In [ ]:
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

envs_grid = AsyncVectorEnv([make_env(i, ObserveType.FULL_GRID, frame_stack=2) for i in range(8)])

agent_grid = DQNGridAgent(
    in_channels=4,
    grid_shape=(20, 20),
    frame_stack=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    device="cuda",
    model_file="../artifacts/models/dqn_grid_v6.pth",
)

stats_grid = agent_grid.train(
    env=envs_grid,
    total_timesteps=1_000_000,  # 8M effective — CNN needs more than vec11
    log_step=5_000,
    buffer_size=300_000,  # larger — long episodes need diverse replay
    batch_size=256,
    gamma=0.997,  # high — apple 50 steps away must still matter
    tau=0.002,  # slow target sync — stable at high apple counts
    eps_init=1.0,
    eps_final=0.03,
    eps_decay=0.25,  # explore for first 250K steps
    learning_starts=5_000,
    train_freq=4,
    gradient_steps=2,
)

Parallel Training:   1%|          | 5255/1000000 [00:05<51:22, 322.72it/s, Avg Rwd (100)=-8.76, Avg Len (100)=3.10, Best=-8.76, Eps=0.981] 